# P5 — Code you can trust, on the course's numbers

**Python primer · session 5**

## What this session is for

The functions you wrote in P2, P3 and P4 work. That is a good start but it is not enough. Code has to be readable long after you wrote it and it has to be usable from a second notebook.

This session has three parts. The first makes your code worth reading again: names with units, constants, small functions and checks. The second puts those habits to work on the numbers of the course: the first detection, the arms of the detectors and the light between the two LIGO sites. The third moves your functions out of the notebook into a library of your own, with a test.

The marks and the temperatures are made up for the exercise. The numbers about GW150914 come from the discovery paper (LIGO Scientific Collaboration and Virgo Collaboration 2016).

## Code worth reading again

Code is read far more often than it is written. The reader is usually you, a long time later, with no memory of the day you wrote it.

Here are two functions as they look when written in a hurry.

In [ ]:
def f(a):
    return sum(a) / len(a)


def h(m, p=18):
    return m >= p


print(f([27, 21, 30]), h(17))

It runs and the answers are right. Now read it as a stranger would. What is `a`: one mark or a list of marks? What is `p`? Does `h` return a number or `True` or `False`? The answers are in your head and your head forgets.

Three changes fix most of this. They are the next three sections.

### Names that say what and in which unit

Put what the value is in the name. `marks` is a list of marks. `pass_mark` is the lowest mark that passes. Where a value has a unit, put the unit in the name too. `temp_c` is a temperature in degrees Celsius. `pause_s` is a time in seconds. Nobody has to hunt for a comment to learn the unit.

Python does not check units. The name is how you and the next reader check them. P1 asked for the unit in a comment. The name is better, because it is there at every place the value is used.

### Constants at the top

A number in the middle of the code is a **bare number**. You see `27` and you must guess what it is. Give each one a name, in capitals, in one cell at the top of the notebook.

Capitals mean a constant: do not change it while the program runs. Python does not enforce this. It is a habit every reader knows.

In [ ]:
import numpy as np
import h5py

# constants: one place, one meaning each
PASS_MARK = 18          # the lowest mark that passes
TOP_MARK = 30           # the highest mark
HIGH_MARK = 27          # above this, a mark is high
HOT_C = 22.0            # degrees Celsius: above this, an hour is hot

Now the two functions again, with the names and the constants.

In [ ]:
def average(marks):
    """Mean of a list of marks."""
    return sum(marks) / len(marks)


def passed(mark, pass_mark=PASS_MARK):
    """True when mark reaches pass_mark."""
    return mark >= pass_mark

The logic has not changed. Each function does one job. Its docstring says what comes back. The default value comes from the constant, so you change a constant once and every function follows.

Call them by name when there is more than one argument:

In [ ]:
print(average([27, 21, 30]))
print(passed(17))
print(passed(mark=17, pass_mark=15))

With one argument a plain call is fine. With two or more, name them, as P2 showed. The call then reads like a sentence and a swapped pair of arguments cannot hide.

### One job per function

Can you describe the function without the word *and*? If not, it has two jobs. A function that computes **and** prints is a common case.

In [ ]:
def show_average(marks):
    print(sum(marks) / len(marks))


result = show_average([27, 21, 30])
print(result)

It prints the average and gives nothing back, so `result` is `None`. You cannot add it to another average. You cannot plot it or test it. `average` returns the number and the caller decides what to do with it.

### Where a line repeats, write a function

Copy and paste is a warning sign. When the same lines appear twice with only a name changed, they want to be a function. Here is the count of high marks, written for two classes.

In [ ]:
class_a = [24, 28, 30, 19, 27, 29, 21]
class_b = [27, 18, 30, 25]

count_a = 0
for mark in class_a:
    if mark > 27:
        count_a = count_a + 1

count_b = 0
for mark in class_b:
    if mark > 27:
        count_b = count_b + 1
print(count_a, count_b)

It works. Now a high mark becomes anything above 26. You must find every `27` in the notebook. If you miss one the count is wrong and Python says nothing.

One definition has one place to fix:

In [ ]:
def count_above(values, threshold=HIGH_MARK):
    """Number of values strictly above the threshold."""
    count = 0
    for value in values:
        if value > threshold:
            count = count + 1
    return count


print(count_above(class_a), count_above(class_b))

### Loading a file

The same advice applies to a file. `example.h5` is the file of temperatures you wrote in P4. This is the quick way to read it.

In [ ]:
f = h5py.File('example.h5', 'r')
d = f['week/temperatures'][:]
r = f['week/temperatures'].attrs['readings_per_hour']
print(len(d) / r)

It works but it has two faults. The file is never closed and `d` and `r` mean nothing. The habit from P4 is better: use `with` and take the number of readings per hour from the file. Put that habit in a function.

In [ ]:
def load_values(path, dataset='week/temperatures'):
    """Values and readings per hour from an HDF5 file."""
    with h5py.File(path, 'r') as f:
        values = f[dataset][:]
        per_hour = f[dataset].attrs['readings_per_hour']
    return values, per_hour


temps_c, per_hour = load_values('example.h5')
print(temps_c.size / per_hour, "hours")
print(count_above(temps_c, threshold=HOT_C), "hours above", HOT_C)

### Small checks, as in P4

P4 showed the `assert`: it does nothing when the condition is `True` and stops when it is `False`. Write one for each function, with an answer you know. Compare floats with `np.isclose`.

In [ ]:
assert np.isclose(average([18, 24, 30]), 24.0)
assert passed(18)
assert not passed(17)
assert count_above([24, 28, 30]) == 2
print("all checks passed")

### The habits, in one list

These are the habits of the first part of this session. Each one is a best practice that makes code easier to read and to trust.

1. Names say what the value is and in which unit.
2. Constants sit at the top, never a bare number in the code.
3. One job per function, with a docstring.
4. Call by name when there is more than one argument.
5. Four spaces, lines under about 80 characters, a blank line between ideas.
6. Comments say why and the code says what.
7. Restart the kernel and run all before you trust a notebook.
8. An `assert` for every function, with an answer you know.
9. Keep data out of the code, in a file beside it.
10. A short commit message that says what changed.

> **The Python style guide**
>
> Most editors can check a file against PEP 8, the Python style guide at <https://peps.python.org/pep-0008/>, as of October 2026.

## The course's numbers

Lecture 0 gave the numbers of GW150914, the first detection. Two black holes of about 36 and 29 solar masses merged 410 megaparsecs away. The signal lasted 0.2 s and its frequency rose from 35 to 250 Hz (LIGO Scientific Collaboration and Virgo Collaboration 2016).

A **solar mass** is the mass of the Sun. A **megaparsec** (Mpc) is a distance used for the sky: 410 Mpc is 1.3 billion light years. A **hertz** (Hz) is one cycle per second.

Each number goes in a variable with its unit in the name, as the first part asked.

In [ ]:
# GW150914, the first detection
mass_1_sun = 36            # solar masses
mass_2_sun = 29            # solar masses
distance_mpc = 410         # megaparsecs: 1.3 billion light years
duration_s = 0.2           # seconds
f_start_hz = 35            # hertz
f_end_hz = 250             # hertz
print(f"The frequency rose by a factor of {f_end_hz / f_start_hz:.0f}")

Seven names for one event is a lot to carry. A dictionary keeps them together under keys, as P3 showed.

In [ ]:
event = {
    "name": "GW150914",
    "mass_1_sun": mass_1_sun,
    "mass_2_sun": mass_2_sun,
    "distance_mpc": distance_mpc,
    "duration_s": duration_s,
    "f_start_hz": f_start_hz,
    "f_end_hz": f_end_hz,
}
print(event["name"], event["distance_mpc"])

An f-string puts the values into a sentence. The keys use single quotes inside a string that uses double quotes.

In [ ]:
print(f"{event['name']}: {event['mass_1_sun']} and "
      f"{event['mass_2_sun']} solar masses, "
      f"{event['distance_mpc']} Mpc away.")
print(f"It lasted {event['duration_s']} s and rose from "
      f"{event['f_start_hz']} to {event['f_end_hz']} Hz.")

The four detectors of lecture 0 are four strings. A list holds them, as in P3.

In [ ]:
detectors = ["LIGO Hanford", "LIGO Livingston", "Virgo", "KAGRA"]
print(len(detectors), "detectors")
for number, name in enumerate(detectors, start=1):
    print(number, name)

## The detectors as data

An arm of a detector has a length. The length is a property of the detector, so it is data. LIGO arms are 4 km long (LIGO Scientific Collaboration 2015) and Virgo arms are 3 km (Virgo Collaboration 2026). A dictionary maps the name of a detector to its arm length in metres.

In [ ]:
arm_length_m = {
    "Hanford": 4000.0,
    "Livingston": 4000.0,
    "Virgo": 3000.0,
}
print(arm_length_m["Virgo"], "m")

Lecture 0 defined the **strain** $h$ as the change in the length of an arm divided by its length, $h = \Delta L / L$. Here $L$ is the length of the arm and $\Delta L$ is how much it changes. A strain has no unit, because it is a ratio of two lengths. Turn it round and the change in length is $\Delta L = h L$.

A strain of 1e-21 is the size of the GW150914 signal (LIGO Scientific Collaboration and Virgo Collaboration 2016). Here is $\Delta L$ for each arm.

In [ ]:
h = 1e-21                       # the strain: no unit
for name, length_m in arm_length_m.items():
    delta_m = h * length_m      # the change in length, in metres
    print(f"{name:11s} {length_m:6.0f} m  moves {delta_m:.1e} m")

The loop does one job but you cannot reuse it. A function can be called from anywhere. Give it a docstring and a default. Call it by name when there is more than one argument, as the first part asked.

In [ ]:
def displacement(h, arm_length_m=4000.0):
    """Change in arm length, in metres, for a strain h."""
    return h * arm_length_m


virgo_m = arm_length_m["Virgo"]
print(f"{displacement(1e-21):.1e} m")
print(f"{displacement(1e-21, arm_length_m=virgo_m):.1e} m")
print(f"{displacement(h=1e-22, arm_length_m=virgo_m):.1e} m")

The second function gives the time light needs to cross a distance. Take the two LIGO sites. Light needs 10 ms to go from one to the other, so they are about 3000 km apart. The two sites recorded GW150914 7 ms apart (LIGO Scientific Collaboration and Virgo Collaboration 2016). An **ms** is a millisecond, a thousandth of a second.

Light travels at 299792.458 km/s (Tiesinga et al. 2021). The unit goes in the name again.

In [ ]:
def time_delay(distance_km, speed_km_s=299792.458):
    """Light travel time, in seconds, over a distance in kilometres."""
    return distance_km / speed_km_s


measured_ms = 7                        # GW150914, between the two sites
light_ms = 1000 * time_delay(3000.0)   # light over 3000 km, in ms
print(f"light needs {light_ms:.1f} ms, the wave arrived {measured_ms} ms apart")
assert measured_ms <= light_ms

The measured delay of 7 ms is below the 10.0 ms that light needs along the line. It cannot be above it. The `assert` keeps that check.

## From a notebook to a library

A function defined in a notebook exists only in that notebook. To use it in a second notebook you would copy it. Copies drift apart. Put it in a file instead. A Python file that you can import is a **module**.

Write `classtools.py` beside this notebook, with the constants and the first function.

In [ ]:
code = '''"""A small library for marks and temperatures."""

PASS_MARK = 18
TOP_MARK = 30
HIGH_MARK = 27


def average(marks):
    """Mean of a list of marks."""
    return sum(marks) / len(marks)
'''
with open('classtools.py', 'w') as f:
    f.write(code)

Mode `'w'` deletes what was there. Mode `'a'` adds to the end, which is what the second function needs.

In [ ]:
more = '''

def passed(mark, pass_mark=PASS_MARK):
    """True when mark reaches pass_mark."""
    return mark >= pass_mark
'''
with open('classtools.py', 'a') as f:
    f.write(more)

with open('classtools.py') as f:
    print(f.read())

### Import it

`import classtools` runs the file and gives you the module. The dot reaches inside, as it does for `math` and `numpy`. `from classtools import average` brings one name in.

In [ ]:
import classtools

print(classtools.average([27, 21, 30]))
print(classtools.passed(17))

In [ ]:
from classtools import average, passed

print(average([18, 24, 30]))
print(passed(mark=18))

### The notebook does not notice a change

Now improve the library: add `count_above` to the file. Does the notebook see it?

In [ ]:
more = '''

def count_above(values, threshold=HIGH_MARK):
    """Number of values strictly above the threshold."""
    count = 0
    for value in values:
        if value > threshold:
            count = count + 1
    return count
'''
with open('classtools.py', 'a') as f:
    f.write(more)

In [ ]:
try:
    print(classtools.count_above([24, 28, 30]))
except AttributeError as error:
    print("the notebook does not know it yet:", error)

Python read the file once, when it was first imported. It keeps that copy in memory. A change to the file does nothing to the copy. There are two ways out. You can restart the kernel or you can call `importlib.reload`.

In [ ]:
import importlib

importlib.reload(classtools)
print(classtools.count_above([24, 28, 30]))

> **Careful — Reload does not reach copied names**
>
> `reload` updates the module. It does not update the names you copied with `from classtools import`. The name `average` in this notebook still points to the old function.
>
> Run the `from` line again after a reload. This is one more reason to prefer `import classtools` and `classtools.average` while you are still changing the code.

### A package

A file beside the notebook works only for notebooks in the same folder. A **package** is a folder that Python can find from anywhere once it is installed. This is the layout:

```text
classtools_pkg/
    pyproject.toml
    test_classtools.py
    classtools/
        __init__.py
```

`classtools_pkg/` is the project folder. The folder `classtools/` inside it is the package. Its `__init__.py` holds the code Python runs on `import classtools`. Here that code is the module you just wrote. The file `pyproject.toml` describes the project.

In [ ]:
import os

os.makedirs('classtools_pkg/classtools', exist_ok=True)

with open('classtools.py') as f:
    source = f.read()

with open('classtools_pkg/classtools/__init__.py', 'w') as f:
    f.write(source)

In [ ]:
config = '''[build-system]
requires = ["setuptools>=61"]
build-backend = "setuptools.build_meta"

[project]
name = "classtools"
version = "0.1.0"
dependencies = []
'''
with open('classtools_pkg/pyproject.toml', 'w') as f:
    f.write(config)

The project has a name and a version. It has no dependencies, because it uses nothing outside Python. The first two lines say how to build it and you can copy them as they are.

Install it from a terminal, inside the project folder:

```bash
cd classtools_pkg
pip install -e .
```

`-e` means *editable*. The install points to your folder instead of copying it, so a change to the code is seen at the next import. The dot means the folder you are in. After this, `import classtools` works from any notebook in the environment.

### Where Python looks

P1 showed `sys.path`. Now it has a meaning. When you write `import classtools`, Python goes through the folders of `sys.path` in order and uses the first match.

In [ ]:
import sys

for folder in sys.path:
    print(repr(folder))

print(classtools.__file__)

The list holds the folders of Python itself and the folder of the notebook. That is why `classtools.py` beside the notebook was found. An installed package is found because `pip` puts it where Python already looks. An editable install puts a link to your folder there.

> **Careful — Same name, two places**
>
> If `classtools.py` stays beside the notebook after you install the package, the file wins. Its folder comes before the installed packages in `sys.path`. When you are not sure which copy runs, print `classtools.__file__`.
>
> Move the file away once the package is installed.

### A test file

A test file collects your checks so that one command runs all of them. A function whose name starts with `test_` is a test. An `assert` inside it is the check. The file is `test_classtools.py`, in the project folder.

In [ ]:
tests = '''import math
from classtools import average, count_above, passed


def test_average():
    assert math.isclose(average([18, 24, 30]), 24.0)
'''
with open('classtools_pkg/test_classtools.py', 'w') as f:
    f.write(tests)

In [ ]:
more = '''

def test_passed():
    assert passed(18)
    assert not passed(17)


def test_count_above():
    assert count_above([24, 28, 30]) == 2
'''
with open('classtools_pkg/test_classtools.py', 'a') as f:
    f.write(more)

Run the tests from the project folder:

```bash
cd classtools_pkg
python -m pytest
```

`pytest` prints one dot for each test that passes. A failing test gets a report with the line and the values. If `pytest` is missing, install it with `pip install pytest`. Run the tests after every change. They are the reason you can change the library without fear.

### Keep it in git

The library is code, so it goes into git. Add a `.gitignore` first. The `__pycache__` folders and the `*.egg-info` folders that Python and pip create do not belong in the history.

```bash
git status
git add classtools_pkg
git commit -m "Add the classtools package and its tests"
git push
```

Now `git log --oneline` tells the story: functions from P2, P3 and P4, readable names, a test, a history and a copy on GitHub.

### A second module: the detector numbers

`displacement` and `time_delay` are worth keeping. Move them out of the notebook into a module, as you did with `classtools`. The module is `gwprimer.py`, beside the notebook.

The first part asked for constants in capitals. The bare numbers `4000.0` and `299792.458` become `ARM_LENGTH_M` and `SPEED_OF_LIGHT_KM_S`. First write the constants and `displacement`.

In [ ]:
code = '''"""Small functions for the detector numbers of the course."""

ARM_LENGTH_M = 4000.0              # one LIGO arm, in metres
SPEED_OF_LIGHT_KM_S = 299792.458   # kilometres per second


def displacement(h, arm_length_m=ARM_LENGTH_M):
    """Change in arm length, in metres, for a strain h."""
    return h * arm_length_m
'''
with open('gwprimer.py', 'w') as f:
    f.write(code)

Mode `'a'` adds to the end of the file, which is what the second function needs.

In [ ]:
more = '''

def time_delay(distance_km, speed_km_s=SPEED_OF_LIGHT_KM_S):
    """Light travel time, in seconds, over a distance in kilometres."""
    return distance_km / speed_km_s
'''
with open('gwprimer.py', 'a') as f:
    f.write(more)

Import it and check both functions against an answer you know, as you did for `classtools`. A strain of 1e-21 on 4000 m is 4e-18 m. Light covers 299792.458 km in one second.

In [ ]:
import gwprimer

assert np.isclose(gwprimer.displacement(1e-21), 4e-18)
assert np.isclose(gwprimer.time_delay(299792.458), 1.0)
print("both checks passed")

Now keep it in git. These are the commands for the terminal, in the folder of the notebook. Do not run them in a cell.

```bash
git status
git add gwprimer.py
git commit -m "Add displacement and time_delay"
git push
```

The module is code and goes in. The strain file is data and stays out, because exercise 2 puts `*.hdf5` in `.gitignore`. To import the module from any folder, make it a package as you did with `classtools`, with `pip install -e .` and a test file. The two `assert` lines above are your first two tests.

## Exercises

### 1. Rewrite a cell worth reading

The cell below converts three temperatures from degrees Celsius to degrees Fahrenheit. Rewrite it. Give the values names with their unit. Write a function `to_fahrenheit(temp_c)` with a docstring. Check it with one `assert`: 100 degrees Celsius is 212 degrees Fahrenheit. Then print the three results with a loop.

In [ ]:
t1 = 18
t2 = 25
t3 = 31
print(t1 * 9 / 5 + 32)
print(t2 * 9 / 5 + 32)
print(t3 * 9 / 5 + 32)

In [ ]:
# your code here

> **Solution**
>
> Typed out together in the session. Write your own version first — the error you
> make on the way is the part you keep.


### 2. Keep data files out of git

Write the file `classtools_pkg/.gitignore` from Python. It must list `*.h5`, `*.hdf5`, `*.wav`, `__pycache__/` and `*.egg-info/`, one per line. Then read the file back and check with `assert` that every pattern is in it.

In [ ]:
# your code here

> **Solution**
>
> Typed out together in the session. Write your own version first — the error you
> make on the way is the part you keep.


### 3. A new function in the library

Add `percent(mark, top_mark=TOP_MARK)` to `classtools.py`. It gives a mark as a percentage of the top mark. Append it to the file and reload the module. Then check that `percent(30)` is 100 and `percent(15)` is 50.

In [ ]:
# your code here

> **Solution**
>
> Typed out together in the session. Write your own version first — the error you
> make on the way is the part you keep.


### 4. A third function in the detector module

Add `strain_from(delta_length_m, arm_length_m=ARM_LENGTH_M)` to `gwprimer.py`. It gives the strain for a change in length, $h = \Delta L / L$. Append it to the file and reload the module. Then check that `strain_from` undoes `displacement`.

This is the bonus. The check is a round trip.

In [ ]:
# your code here

> **Solution**
>
> Typed out together in the session. Write your own version first — the error you
> make on the way is the part you keep.


## Where this leaves you

You can write a function that a stranger could read, with the unit in every name. You can check it with an `assert`. You have used those habits on the numbers of the course: the arms move by $4\times10^{-18}$ m and the wave cannot arrive later than light. You can import your functions from any notebook as a module or a package, with a test that runs in one command.

**Do.** Pick two functions from P2, P3 or P4 that you still like. Move them into `classtools`, write one test for each and commit the result.

### References

**References**

- LIGO Scientific Collaboration and Virgo Collaboration (2016). *Observation of Gravitational Waves from a Binary Black Hole Merger*.
- LIGO Scientific Collaboration (2015). *Advanced LIGO*.
- Virgo Collaboration (2026). *Advanced Virgo during the LIGO-Virgo-KAGRA fourth observing run*.
- Tiesinga et al. (2021). *CODATA recommended values of the fundamental physical constants: 2018*.